# MNQ – Hypothesis 3: Time‑of‑Day Short (16:00 ET) – IS optimization, OOS validation

## 1. Setup and load IS data (regular hours)

In [13]:
import pandas as pd
import numpy as np
from itertools import product
import os
import warnings
warnings.filterwarnings('ignore')

IS_PATH = "../../../../Local-Data/processed/MNQ_IS_1min.csv"
OOS_TICKS_PATH = "../../../../Local-Data/processed/MNQ_OOS_ticks.csv"

df_is = pd.read_csv(IS_PATH, parse_dates=['DateTime'], index_col='DateTime')
print(f"IS data: {len(df_is):,} rows")

# Filter regular hours (8:30-16:00)
start_time = pd.Timestamp('2026-02-01 08:30:00').time()
end_time   = pd.Timestamp('2026-02-01 16:00:00').time()
df_is_reg = df_is.between_time(start_time, end_time).copy()
print(f"IS regular hours rows: {len(df_is_reg):,}")


IS data: 87,430 rows
IS regular hours rows: 28,699


## 2. Backtest function (short at 16:00)

In [14]:
def backtest_tod_short(df, entry_hour=16, entry_minute=0, hold_minutes=15, tolerance_minutes=5):
    """
    Short at specified hour:minute. If exact time not found, use the next available bar within tolerance.
    Exit after hold_minutes bars.
    """
    trades = []
    for date, day_df in df.groupby(df.index.date):
        target = pd.Timestamp(date).replace(hour=entry_hour, minute=entry_minute)
        # Find the nearest bar not earlier than target, within tolerance
        future_bars = day_df[day_df.index >= target]
        if future_bars.empty:
            continue
        entry_time = future_bars.index[0]
        # Check if the time difference is within tolerance
        diff_minutes = (entry_time - target).total_seconds() / 60
        if diff_minutes > tolerance_minutes:
            continue
        entry_idx = day_df.index.get_loc(entry_time)
        if entry_idx + hold_minutes >= len(day_df):
            continue
        exit_dt = day_df.index[entry_idx + hold_minutes]
        entry_price = day_df.loc[entry_time, 'Open']
        exit_price = day_df.loc[exit_dt, 'Close']
        pnl_points = entry_price - exit_price   # short
        trades.append(pnl_points * 2.0)         # MNQ point = $2
    return trades


## 3. Grid search on IS

In [15]:
entry_hours = [15, 15, 15, 15, 15, 16]   # we can test 15:30, 15:40, 15:50, 15:55, 16:00
entry_minutes = [30, 40, 50, 55, 59, 0]
hold_minutes = [5, 10, 15, 20, 25, 30]

results = []
for eh, em in zip(entry_hours, entry_minutes):
    for h in hold_minutes:
        trades = backtest_tod_short(df_is_reg, eh, em, h, tolerance_minutes=5)
        if trades:
            total = sum(trades)
            wins = sum(1 for t in trades if t > 0)
            win_rate = wins / len(trades)
            gross_profit = sum(t for t in trades if t > 0)
            gross_loss = -sum(t for t in trades if t < 0)
            pf = gross_profit / gross_loss if gross_loss > 0 else 999
            results.append({
                'entry_time': f"{eh:02d}:{em:02d}",
                'hold': h,
                'trades': len(trades),
                'pnl_usd': total,
                'win_rate': win_rate,
                'profit_factor': pf
            })
        else:
            print(f"No trades for entry {eh:02d}:{em:02d}, hold {h}")

if results:
    results_df = pd.DataFrame(results).sort_values('profit_factor', ascending=False)
    print("\nTop 10 time‑of‑day strategies on IS:")
    print(results_df.head(10).to_string(index=False))
else:
    print("No trades generated. Check data range.")
    import sys
    sys.exit()

No trades for entry 15:40, hold 25
No trades for entry 15:40, hold 30
No trades for entry 15:50, hold 15
No trades for entry 15:50, hold 20
No trades for entry 15:50, hold 25
No trades for entry 15:50, hold 30
No trades for entry 15:55, hold 10
No trades for entry 15:55, hold 15
No trades for entry 15:55, hold 20
No trades for entry 15:55, hold 25
No trades for entry 15:55, hold 30
No trades for entry 15:59, hold 5
No trades for entry 15:59, hold 10
No trades for entry 15:59, hold 15
No trades for entry 15:59, hold 20
No trades for entry 15:59, hold 25
No trades for entry 15:59, hold 30
No trades for entry 16:00, hold 5
No trades for entry 16:00, hold 10
No trades for entry 16:00, hold 15
No trades for entry 16:00, hold 20
No trades for entry 16:00, hold 25
No trades for entry 16:00, hold 30

Top 10 time‑of‑day strategies on IS:
entry_time  hold  trades  pnl_usd  win_rate  profit_factor
     15:30     5      63    338.5  0.555556       1.228408
     15:40    10      63    179.0  0.4761

## 4. Validate top 3 on OOS

In [16]:
if os.path.exists(OOS_TICKS_PATH):
    # Load OOS ticks and aggregate to 1‑min
    df_ticks = pd.read_csv(OOS_TICKS_PATH, parse_dates=['DateTime'], index_col='DateTime')
    ohlc = df_ticks['Price'].resample('1min').ohlc()
    vol = df_ticks['Volume'].resample('1min').sum()
    ohlc.columns = ['Open', 'High', 'Low', 'Close']
    df_oos = ohlc.join(vol).rename(columns={'Volume': 'Volume'}).dropna()
    # Filter regular hours
    df_oos_reg = df_oos.between_time(start_time, end_time).copy()
    print(f"OOS regular hours rows: {len(df_oos_reg):,}")

    top3 = results_df.head(3).copy()
    valid_results = []
    for idx, row in top3.iterrows():
        trades = backtest_tod_short(df_oos_reg, 16, 0, int(row['hold']))
        if trades:
            total = sum(trades)
            wins = sum(1 for t in trades if t > 0)
            win_rate = wins / len(trades)
            gross_profit = sum(t for t in trades if t > 0)
            gross_loss = -sum(t for t in trades if t < 0)
            pf = gross_profit / gross_loss if gross_loss > 0 else 999
            valid_results.append({
                'hold': row['hold'], 
                'trades_oos': len(trades), 
                'pnl_usd_oos': total,
                'win_rate_oos': win_rate, 
                'profit_factor_oos': pf
            })
    if valid_results:
        valid_df = pd.DataFrame(valid_results).sort_values('profit_factor_oos', ascending=False)
        print("\n=== OOS Validation Results (May 2026) ===")
        print(valid_df.to_string(index=False))
        
        best = valid_df.iloc[0]
        print("\n=== FINAL SELECTED PARAMETERS ===")
        print(f"Holding minutes: {int(best['hold'])}")
        print(f"Trades in OOS: {best['trades_oos']}")
        print(f"Profit factor (OOS): {best['profit_factor_oos']:.2f}")
        print(f"Net profit (USD, 1 contract): ${best['pnl_usd_oos']:.2f}")
    else:
        print("No trades generated on OOS. Strategy not viable.")
else:
    print("OOS ticks file not found.")

OOS regular hours rows: 8,118
No trades generated on OOS. Strategy not viable.


## Hypothesis 1 – Negative Gap Reversal (MNQ) – Conclusion

- **Best OOS Profit Factor:** 2.58  
- **Trades in May (OOS):** 8  
- **Optimal parameters:** Gap threshold = 0.025%, hold = 15 min, ATR > 70th percentile, no stop  
- **Expected daily trades:** ≈ 0.4  

**Verdict:** Profitable but low frequency. High risk of zero signals on competition day. **Not recommended as primary strategy** – may be used as a low‑risk complement.